# Módulo 4: Serving, Analytics e ML (Exercícios)

Cinco exercícios curtos, cobrindo as quatro partes do módulo:

1. **BI e camada semântica** (exercício 1)
2. **Disponibilização para ML e Feature Store** (exercício 2)
3. **Reverse ETL** (exercícios 3 e 4)
4. **Governança e qualidade** (exercício 5)

Cada exercício tem uma célula para a sua resposta e, logo depois, uma célula **"Confira sua resposta"**. Se ela imprimir `✅`, a resposta está certa; se der erro, revise o código.

Este notebook usa o mesmo Data Warehouse dos módulos anteriores (`data/modulo_03/data_warehouse.duckdb`), aberto em **modo somente leitura**: nada do que você fizer aqui altera o banco.

## Configuração de Acesso aos Dados (Google Colab / Local)

> **No Google Colab:** certifique-se de que a pasta `data` do repositório foi enviada para dentro de **`Meu Drive/Colab Notebooks/`** antes de executar a célula abaixo.  
> **Em execução local (VS Code / Jupyter):** o código ajusta automaticamente o diretório para a raiz do repositório.


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    source = '/content/drive/MyDrive/Colab Notebooks'
    os.chdir(source)
except ImportError:
    # Execução local (VS Code / Jupyter): ajusta o diretório para a raiz do repositório
    if Path('../../../data').exists():
        os.chdir(Path('../../../').resolve())

print("Diretório atual:", os.getcwd())
print("Pastas em data/:", os.listdir('data'))


In [1]:
import duckdb
import pandas as pd
import json
import hashlib
import os
from datetime import datetime, timedelta

# Banco em memória para os exercícios, com o DW "anexado" em modo somente leitura.
# As tabelas do DW são acessadas com o prefixo dw. (por exemplo, dw.fato_vendas)
con = duckdb.connect()
con.execute("ATTACH 'data/modulo_03/data_warehouse.duckdb' AS dw (READ_ONLY)")

display(con.execute("SHOW ALL TABLES").df()[["database", "name"]].query("database == 'dw'"))

,database,name
0,dw,dim_cliente
1,dw,dim_data
2,dw,dim_produto
3,dw,fato_vendas
4,dw,mart_comercial
5,dw,mart_gestao
6,dw,resumo_vendas_categoria
7,dw,vendas_raw
8,dw,vendas_tratadas
9,dw,vw_metricas_vendas


## Parte 1: BI e camada semântica

### Exercício 1: camada semântica, qual é "o faturamento"?

Um dashboard mostra o faturamento **antes** dos descontos; outro mostra **depois**. Os dois números estão "certos", mas são diferentes, e a diretoria não sabe em qual acreditar.

Crie a view `vw_metricas_receita`, **por categoria**, que deixe as três métricas explícitas e com nomes sem ambiguidade:

- `faturamento_bruto`: soma de `quantidade * preco_unitario`;
- `desconto_total`: soma de `desconto`;
- `faturamento_liquido`: soma de `valor_total`.

**Para discutir:** depois de rodar, confira que bruto menos desconto é igual ao líquido. Qual dos dois o time deveria chamar só de "faturamento"? Por que essa decisão precisa ficar registrada num único lugar?

In [ ]:
sql_ex1 = """
-- CREATE OR REPLACE VIEW vw_metricas_receita AS ...
"""

con.execute(sql_ex1)
display(con.execute("SELECT * FROM vw_metricas_receita ORDER BY categoria").df())

In [ ]:
# Confira sua resposta (não altere esta célula)
df_ex1 = con.execute("SELECT * FROM vw_metricas_receita").df()
colunas = {"categoria", "faturamento_bruto", "desconto_total", "faturamento_liquido"}
assert colunas.issubset(df_ex1.columns), f"A view precisa ter as colunas {colunas}"
diferenca = (df_ex1["faturamento_bruto"] - df_ex1["desconto_total"] - df_ex1["faturamento_liquido"]).abs().max()
assert diferenca == 0, "Bruto menos desconto deveria ser igual ao líquido"
assert df_ex1["faturamento_bruto"].sum() == 17590, "O faturamento bruto total deveria ser 17590"
print("Exercício 1 correto")

## Parte 2: Disponibilização para ML e Feature Store

### Exercício 2: criando novas features por cliente

O time de Ciência de Dados pediu duas features novas, com **uma linha por cliente**:

- `dias_como_cliente`: número de dias entre a **primeira** e a **última** compra do cliente;
- `pct_gasto_eletronicos`: fração do gasto do cliente que foi em Eletrônicos (de 0 a 1, arredondada em 2 casas).

A tabela final deve ter as colunas `id_cliente`, `dias_como_cliente` e `pct_gasto_eletronicos`.

*Dicas:* `DATEDIFF('day', MIN(data), MAX(data))` e `SUM(CASE WHEN categoria = 'Eletrônicos' THEN valor_total ELSE 0 END)`.

In [ ]:
sql_ex2 = """
-- escreva sua consulta aqui
"""

df_ex2 = con.execute(sql_ex2).df()
display(df_ex2)

In [ ]:
# Confira sua resposta (não altere esta célula)
assert len(df_ex2) == 8, "Deveria haver uma linha por cliente (8 clientes)"
por_cliente = df_ex2.set_index("id_cliente")
assert por_cliente.loc[101, "dias_como_cliente"] == 3, "Ana Silva (101): 3 dias entre a primeira e a última compra"
assert por_cliente.loc[102, "dias_como_cliente"] == 26, "Carlos Souza (102): 26 dias entre a primeira e a última compra"
assert por_cliente.loc[104, "pct_gasto_eletronicos"] == 1.0, "João Santos (104) só comprou Eletrônicos"
assert por_cliente.loc[102, "pct_gasto_eletronicos"] == 0.0, "Carlos Souza (102) não comprou Eletrônicos"
print("Exercício 2 correto")

## Parte 3: Reverse ETL

Nesta parte não há API real: os exercícios testam a **lógica** que fica por trás de uma sincronização segura com outras aplicações.

### Exercício 3: tentar de novo ou desistir?

Ao enviar dados para a API de um CRM, cada resposta tem um código de status HTTP. Escreva a função `classificar_status(status)`, que devolve:

- `"sucesso"` para códigos de 200 a 299;
- `"temporario"` para 429 e para códigos de 500 a 599 (vale tentar de novo, com espera);
- `"permanente"` para qualquer outro código (repetir não resolve, então o registro vai para a lista de falhas).

In [ ]:
def classificar_status(status):
    # escreva sua função aqui
    pass

In [ ]:
# Confira sua resposta (não altere esta célula)
casos = {200: "sucesso", 201: "sucesso", 429: "temporario", 500: "temporario",
         503: "temporario", 400: "permanente", 401: "permanente", 404: "permanente"}
for status, esperado in casos.items():
    obtido = classificar_status(status)
    assert obtido == esperado, f"Status {status}: esperado '{esperado}', obtido '{obtido}'"
print("Exercício 3 correto")

### Exercício 4: cada campo tem um único dono

O dicionário `DONO_DOS_CAMPOS` diz qual sistema produz cada campo. Escreva a função `campos_para_sync(dono_dos_campos)`, que devolve, **em ordem alfabética**, apenas os campos que o reverse ETL pode enviar ao CRM.

**Para discutir:** o que aconteceria, a cada ciclo de sincronização, se `atualizado_em` fosse incluído no envio?

In [ ]:
DONO_DOS_CAMPOS = {
    "segmento": "dw",
    "faturamento_total": "dw",
    "ticket_medio": "dw",
    "atualizado_em": "crm",
    "anotacao_vendedor": "crm",
    "status_atendimento": "crm",
}


def campos_para_sync(dono_dos_campos):
    # escreva sua função aqui
    pass

In [ ]:
# Confira sua resposta (não altere esta célula)
assert campos_para_sync(DONO_DOS_CAMPOS) == ["faturamento_total", "segmento", "ticket_medio"], \
    "Devolva só os campos cujo dono é o DW, em ordem alfabética"
print(" Exercício 4 correto")

## Parte 4: Governança, qualidade e contratos

A função `validar` abaixo é a mesma da seção de governança do módulo: ela confere **nulidade** e **limites aceitáveis**. No exercício, você vai acrescentar novas regras ao contrato.

In [ ]:
def validar(df, regras):
    problemas = []

    # Regra 1: nulidade
    for coluna in df.columns:
        pct_nulos = df[coluna].isna().mean()
        if pct_nulos > regras["max_nulos"]:
            problemas.append(f"Coluna '{coluna}': {pct_nulos:.0%} de valores vazios (limite: {regras['max_nulos']:.0%})")

    # Regra 2: limites aceitáveis
    for coluna, (minimo, maximo) in regras["limites"].items():
        fora = df[(df[coluna] < minimo) | (df[coluna] > maximo)]
        if len(fora) > 0:
            problemas.append(f"Coluna '{coluna}': {len(fora)} valor(es) fora do intervalo de {minimo} a {maximo}")

    return problemas


# Data mart usado nos testes: segmentos de clientes calculados a partir do mart_gestao
mart_segmentos = con.execute("""
    SELECT
        id_cliente,
        CASE
            WHEN faturamento_total >= 3000 THEN 'VIP'
            WHEN total_vendas >= 2          THEN 'recorrente'
            ELSE 'ocasional'
        END AS segmento,
        faturamento_total
    FROM dw.mart_gestao
""").df()

display(mart_segmentos)

### Exercício 5: unicidade e valores permitidos

Escreva a função `validar_v2(df, regras)`, que:

1. chama a `validar` original (nulidade e limites) e aproveita os problemas que ela encontrar;
2. **unicidade:** se houver valores repetidos na coluna `regras["chave_unica"]`, adiciona um problema com a palavra **"duplicado"**;
3. **valores permitidos:** para cada coluna em `regras["valores_permitidos"]`, se houver algum valor fora do conjunto permitido, adiciona um problema com as palavras **"não permitido"**.

*Dicas:* `df[coluna].duplicated().sum()` e `~df[coluna].isin(conjunto)`.

In [ ]:
regras_segmentos = {
    "max_nulos": 0.0,
    "limites": {"faturamento_total": (0, 100_000)},
    "chave_unica": "id_cliente",
    "valores_permitidos": {"segmento": {"VIP", "recorrente", "ocasional"}},
}


def validar_v2(df, regras):
    problemas = validar(df, regras)
    # acrescente aqui as regras de unicidade e de valores permitidos
    return problemas

In [ ]:
# Confira sua resposta (não altere esta célula)
# Caso de sucesso: o data mart real deve passar
assert validar_v2(mart_segmentos, regras_segmentos) == [], "O data mart real deveria ser aprovado"

# Caso de falha: uma linha duplicada e um segmento inventado
mart_com_erro = pd.concat([mart_segmentos, mart_segmentos.iloc[[0]]], ignore_index=True)
mart_com_erro.loc[1, "segmento"] = "Ouro"
problemas = validar_v2(mart_com_erro, regras_segmentos)
for p in problemas:
    print("  -", p)
assert len(problemas) == 2, "Deveriam ser encontrados exatamente 2 problemas"
assert any("duplicado" in p for p in problemas), "Faltou detectar a duplicata"
assert any("não permitido" in p for p in problemas), "Faltou detectar o segmento inválido"
print("Exercício 5 correto")

In [ ]:
con.close()
print("Fim dos exercícios.")